# 🔌 Projeto: Engenharia de Dados em Big Data
## Onde Construir um Datacenter no Brasil?
### Etapa 3 — Aplicação de ML e Treinamento de Modelos

**Disciplina:** Fundamentos de Dados e Analytics — Engenharia de Dados em Big Data  
**Profs.:** Fabio Rossi Versolatto · Gustavo Moreira Calixto

> **Pré-requisito:** Etapa 2 concluída — `gold_empreendimentos_features` populada no MongoDB.  
> Esta etapa carrega exclusivamente dados do MongoDB para treinar e avaliar os modelos.

---

## 0.1 — Instalação de dependências

In [ ]:
!pip install -q pymongo dnspython plotly xgboost openpyxl

## 0.2 — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from datetime import datetime
from pymongo import MongoClient, ASCENDING
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, mean_absolute_error,
    r2_score, mean_squared_error,
)
from xgboost import XGBClassifier, XGBRegressor
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (12, 5)
print('✅ Imports prontos.')

## 0.3 — Conexão MongoDB Atlas

> **Pré-requisito:** secret `MONGO_URI` cadastrado em *Colab → 🔑 Secrets*

In [ ]:
from google.colab import userdata

MONGO_URI = userdata.get('MONGO_URI')
assert MONGO_URI, '❌ Cadastre o secret MONGO_URI em Configurações → Secrets'

client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=15000)
client.admin.command('ping')
db = client['aneel_datacenters_v2']
print('✅ Conectado ao MongoDB Atlas. Banco:', db.name)
print('📋 Coleções:', sorted(db.list_collection_names()))

---
# 🟩 ETAPA 3 — Aplicação de ML e Treinamento de Modelos

Dois problemas integrados:
1. **Classificação multiclasse** — `DscFaseUsina` ∈ {Operação · Construção · Construção não iniciada}
2. **Regressão** — `MdaPotenciaFiscalizadaKw`

## 3.1 — Carregamento da Gold (MongoDB) e preparação do dataset ML

In [ ]:
# ── Carrega gold_empreendimentos_features do MongoDB ─────────────────────────
print('⏳ Carregando gold_empreendimentos_features...')
emp = pd.DataFrame(list(
    db['gold_empreendimentos_features'].find({}, {'_id': 0})
))
assert len(emp) > 0, (
    '❌ gold_empreendimentos_features está vazia.\n'
    'Execute a Etapa 2 antes de prosseguir.'
)
print(f'✅ Dataset carregado: {emp.shape}')
emp.head(3)

In [ ]:
# ── Filtra as 3 fases mais representativas ───────────────────────────────────
df_ml = emp.dropna(subset=[
    'DscFaseUsina', 'MdaPotenciaOutorgadaKw',
    'SigTipoGeracao', 'SigUFPrincipal', 'DscOrigemCombustivel'
]).copy()

top_fases = df_ml['DscFaseUsina'].value_counts().head(3).index
df_ml = df_ml[df_ml['DscFaseUsina'].isin(top_fases)]
print('Distribuição da target (fase):')
print(df_ml['DscFaseUsina'].value_counts())

In [ ]:
# ── Feature engineering ──────────────────────────────────────────────────────
df_ml['ano_operacao'] = (
    pd.to_datetime(df_ml.get('DatEntradaOperacao'), errors='coerce')
    .dt.year.fillna(2030)
)
df_ml['log_potencia']    = np.log1p(df_ml['MdaPotenciaOutorgadaKw'].clip(lower=0))
df_ml['eh_renovavel_int'] = df_ml['eh_renovavel'].astype(int)

# Imputa nulos com mediana global
COLS_CLIMA = [
    'inmet_temp_media', 'inmet_temp_std', 'inmet_umid_media',
    'inmet_prec_total', 'inmet_radiacao_med', 'inmet_vento_med',
    'inmet_dist_km', 'dc_proximos', 'anatel_total',
]
for c in COLS_CLIMA:
    if c in df_ml.columns:
        df_ml[c] = df_ml[c].fillna(df_ml[c].median())

FEATURES_CAT = ['SigTipoGeracao', 'SigUFPrincipal',
                'DscOrigemCombustivel', 'DscTipoOutorga']
FEATURES_NUM = [
    'MdaPotenciaOutorgadaKw', 'log_potencia', 'ano_operacao', 'eh_renovavel_int',
    'lat', 'lon',
    'inmet_temp_media', 'inmet_temp_std', 'inmet_umid_media',
    'inmet_prec_total', 'inmet_radiacao_med', 'inmet_vento_med', 'inmet_dist_km',
    'dc_proximos', 'anatel_total',
]
FEATURES_CAT = [c for c in FEATURES_CAT if c in df_ml.columns]
FEATURES_NUM = [c for c in FEATURES_NUM if c in df_ml.columns]

encoders = {}
for c in FEATURES_CAT:
    le = LabelEncoder()
    df_ml[c + '_enc'] = le.fit_transform(df_ml[c].astype(str))
    encoders[c] = le

X = df_ml[[c + '_enc' for c in FEATURES_CAT] + FEATURES_NUM]

le_target = LabelEncoder().fit(df_ml['DscFaseUsina'])
y_class   = le_target.transform(df_ml['DscFaseUsina'])
print('X shape:', X.shape, '| Classes:', list(le_target.classes_))

## 3.2 — Modelo de Classificação (fase do empreendimento)

Random Forest e XGBoost treinados com validação estratificada.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_class, test_size=0.25, random_state=42, stratify=y_class
)

# ── Random Forest ─────────────────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=300, max_depth=15,
    class_weight='balanced', n_jobs=-1, random_state=42
)
rf.fit(X_tr, y_tr)
pred_rf = rf.predict(X_te)

# ── XGBoost ───────────────────────────────────────────────────────────────────
xgb = XGBClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.1,
    eval_metric='mlogloss', n_jobs=-1, random_state=42
)
xgb.fit(X_tr, y_tr)
pred_xgb = xgb.predict(X_te)

print(f'Random Forest — Accuracy: {accuracy_score(y_te, pred_rf):.4f}')
print(f'XGBoost       — Accuracy: {accuracy_score(y_te, pred_xgb):.4f}')
print('\n=== Classification Report (XGBoost) ===')
print(classification_report(y_te, pred_xgb, target_names=le_target.classes_))

In [ ]:
# ── Matriz de confusão + importância das features ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

cm = confusion_matrix(y_te, pred_xgb)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=le_target.classes_,
            yticklabels=le_target.classes_)
axes[0].set_title('Matriz de confusão — XGBoost')
axes[0].set_xlabel('Predito')
axes[0].set_ylabel('Real')

imp = pd.Series(xgb.feature_importances_, index=X.columns).sort_values()
imp.plot(kind='barh', ax=axes[1], color='teal')
axes[1].set_title('Importância das features — XGBoost')

plt.tight_layout()
plt.show()

## 3.3 — Modelo de Regressão (potência fiscalizada)

Random Forest treinado em escala log para `MdaPotenciaFiscalizadaKw`.

In [ ]:
df_reg = df_ml.dropna(subset=['MdaPotenciaFiscalizadaKw'])
df_reg = df_reg[df_reg['MdaPotenciaFiscalizadaKw'] > 0]

X_reg = df_reg[
    [c + '_enc' for c in FEATURES_CAT] +
    [c for c in FEATURES_NUM if c != 'MdaPotenciaOutorgadaKw'] +
    ['MdaPotenciaOutorgadaKw']
]
y_reg = np.log1p(df_reg['MdaPotenciaFiscalizadaKw'])

Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=42
)

rf_reg = RandomForestRegressor(
    n_estimators=400, max_depth=20, n_jobs=-1, random_state=42
)
rf_reg.fit(Xr_tr, yr_tr)
pred_reg = rf_reg.predict(Xr_te)

y_true = np.expm1(yr_te)
y_pred = np.expm1(pred_reg)
print(f'R²   : {r2_score(yr_te, pred_reg):.4f}')
print(f'MAE  : {mean_absolute_error(y_true, y_pred):,.0f} kW')
print(f'RMSE : {np.sqrt(mean_squared_error(y_true, y_pred)):,.0f} kW')

In [ ]:
# ── Gráfico Real × Predito ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_true, y_pred, alpha=0.3, s=10)
lim = [max(float(y_true.min()), 1), float(y_true.max())]
ax.plot(lim, lim, 'r--')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Potência fiscalizada real (kW)')
ax.set_ylabel('Potência fiscalizada predita (kW)')
ax.set_title('Random Forest — Real × Predito (escala log)')
plt.tight_layout()
plt.show()

## 3.4 — 🏆 Índice de Atratividade para Datacenter

Combina saídas dos modelos + features climáticas + telecom em um score único [0–1]:

| Componente                         | Peso | Direção |
|------------------------------------|------|---------|
| Capacidade futura prevista (kW)    | 0,35 | +       |
| % renovável da matriz local        | 0,15 | +       |
| Cobertura telecom (ANATEL)         | 0,15 | +       |
| Estabilidade térmica (1/temp_std)  | 0,10 | +       |
| Inverso de precipitação extrema    | 0,10 | +       |
| Inverso de saturação (DCs próximos)| 0,15 | +       |

In [ ]:
# ── Carrega silver_anatel_uf para o índice ────────────────────────────────────
silver_anatel = pd.DataFrame(list(
    db['silver_anatel_uf'].find({}, {'_id': 0})
))

# Aplica regressor e classificador a toda a base
gold_ml = df_ml.copy()
X_full = gold_ml[
    [c + '_enc' for c in FEATURES_CAT] +
    [c for c in FEATURES_NUM if c != 'MdaPotenciaOutorgadaKw'] +
    ['MdaPotenciaOutorgadaKw']
]
gold_ml['potencia_fiscalizada_predita_kw'] = np.expm1(rf_reg.predict(X_full))

proba_xgb = xgb.predict_proba(X_full[X.columns])
op_idx = (
    list(le_target.classes_).index('Operação')
    if 'Operação' in le_target.classes_ else 0
)
gold_ml['prob_operacao'] = proba_xgb[:, op_idx]

def norm01(s):
    s = s.astype(float)
    if s.max() == s.min():
        return s * 0
    return (s - s.min()) / (s.max() - s.min())

agg_uf = gold_ml.groupby('SigUFPrincipal').agg(
    cap_fiscalizada_predita_MW = ('potencia_fiscalizada_predita_kw', lambda x: x.sum() / 1000),
    prob_operacao_media        = ('prob_operacao', 'mean'),
    pct_renovavel              = ('eh_renovavel_int', lambda x: x.mean() * 100),
    temp_std_med               = ('inmet_temp_std', 'mean'),
    prec_max_med               = ('inmet_prec_max_h', 'mean'),
    dc_saturacao               = ('dc_proximos', 'mean'),
    qtd_empreendimentos        = ('CodCEG', 'count'),
).reset_index()

agg_uf = agg_uf.merge(
    silver_anatel[['UF', 'anatel_total']],
    left_on='SigUFPrincipal', right_on='UF', how='left'
).drop(columns=['UF'])
agg_uf['anatel_total'] = agg_uf['anatel_total'].fillna(0)

# Normalização dos componentes
agg_uf['s_capacidade'] = norm01(agg_uf['cap_fiscalizada_predita_MW'])
agg_uf['s_renovavel']  = norm01(agg_uf['pct_renovavel'])
agg_uf['s_telecom']    = norm01(agg_uf['anatel_total'])
agg_uf['s_estab_term'] = 1 - norm01(agg_uf['temp_std_med'].fillna(agg_uf['temp_std_med'].median()))
agg_uf['s_baixa_prec'] = 1 - norm01(agg_uf['prec_max_med'].fillna(agg_uf['prec_max_med'].median()))
agg_uf['s_baixa_sat']  = 1 - norm01(agg_uf['dc_saturacao'])

agg_uf['indice_atratividade_dc'] = (
    0.35 * agg_uf['s_capacidade'] +
    0.15 * agg_uf['s_renovavel']  +
    0.15 * agg_uf['s_telecom']    +
    0.10 * agg_uf['s_estab_term'] +
    0.10 * agg_uf['s_baixa_prec'] +
    0.15 * agg_uf['s_baixa_sat']
).round(4)

ranking = agg_uf.sort_values('indice_atratividade_dc', ascending=False).reset_index(drop=True)
ranking.index += 1
print('🏆 RANKING DE UFs — ATRATIVIDADE PARA DATACENTER')
display(ranking[['SigUFPrincipal', 'indice_atratividade_dc',
                 'cap_fiscalizada_predita_MW', 'pct_renovavel',
                 'anatel_total', 'temp_std_med',
                 'prec_max_med', 'dc_saturacao']].head(15))

In [ ]:
# ── Top 15 UFs ────────────────────────────────────────────────────────────────
fig = px.bar(
    ranking.head(15),
    x='SigUFPrincipal', y='indice_atratividade_dc',
    color='indice_atratividade_dc', color_continuous_scale='viridis',
    text='indice_atratividade_dc',
    title='Top 15 UFs — Índice de Atratividade para Datacenters',
    labels={'indice_atratividade_dc': 'Índice (0–1)', 'SigUFPrincipal': 'UF'},
)
fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.update_layout(yaxis_range=[0, 1.05])
fig.show()

In [ ]:
# ── Decomposição dos componentes — Top 5 ─────────────────────────────────────
top5 = ranking.head(5)
comp = top5.set_index('SigUFPrincipal')[[
    's_capacidade', 's_renovavel', 's_telecom',
    's_estab_term', 's_baixa_prec', 's_baixa_sat'
]]
ax = comp.plot(kind='bar', stacked=True, figsize=(10, 5),
               colormap='viridis', width=0.7)
ax.set_title('Decomposição do Índice de Atratividade — Top 5 UFs')
ax.set_ylabel('Contribuição normalizada')
ax.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.tight_layout()
plt.show()

In [ ]:
# ── Mapa final ────────────────────────────────────────────────────────────────
mapa_emp = emp.dropna(subset=['lat', 'lon']).merge(
    ranking[['SigUFPrincipal', 'indice_atratividade_dc']],
    on='SigUFPrincipal', how='left'
)
mapa_emp = mapa_emp[
    mapa_emp['DscFaseUsina'].isin(['Operação', 'Construção'])
].sample(min(4000, len(mapa_emp)), random_state=42)

fig = px.scatter_mapbox(
    mapa_emp, lat='lat', lon='lon',
    color='indice_atratividade_dc',
    color_continuous_scale='viridis',
    size='MdaPotenciaOutorgadaKw',
    hover_name='NomEmpreendimento',
    hover_data={
        'SigUFPrincipal': True, 'DscFaseUsina': True,
        'MdaPotenciaOutorgadaKw': ':.0f',
        'indice_atratividade_dc': ':.3f',
    },
    zoom=3, height=650,
    title='Mapa Final — Onde construir um datacenter? (cor = atratividade da UF)',
)
fig.update_layout(mapbox_style='open-street-map',
                  margin=dict(r=0, t=40, l=0, b=0))
fig.show()

## 3.5 — Persistência das predições e do ranking final (Gold)

In [ ]:
# ── gold_predicoes_potencia ───────────────────────────────────────────────────
COLS_PRED = [
    'CodCEG', 'NomEmpreendimento', 'SigUFPrincipal',
    'SigTipoGeracao', 'DscFaseUsina',
    'MdaPotenciaFiscalizadaKw', 'potencia_fiscalizada_predita_kw',
    'prob_operacao', 'lat', 'lon',
]
saida_pred = gold_ml[[c for c in COLS_PRED if c in gold_ml.columns]].copy()

col_pred = db['gold_predicoes_potencia']
col_pred.drop()
col_pred.insert_many(saida_pred.to_dict(orient='records'))
print(f'✅ gold_predicoes_potencia: {col_pred.estimated_document_count():,} predições')

# ── gold_ranking_atratividade_dc ──────────────────────────────────────────────
col_rk = db['gold_ranking_atratividade_dc']
col_rk.drop()
col_rk.insert_many(ranking.to_dict(orient='records'))
print(f'✅ gold_ranking_atratividade_dc: {col_rk.estimated_document_count()} UFs')

print('\n📋 Coleções Gold finais:',
      [c for c in db.list_collection_names() if c.startswith('gold_')])

In [ ]:
# Verificação final
print('📋 Todas as coleções no MongoDB:')
for c in sorted(db.list_collection_names()):
    n = db[c].estimated_document_count()
    print(f'   {c:45s} {n:>10,} docs')

---
## ✅ Conclusões

### 🔧 Engenharia de Dados
- **Padrão Medallion** com sucesso: 4 fontes heterogêneas → Bronze → Silver → Gold no MongoDB.
- **Join geoespacial via KDTree** enriqueceu ~24 mil empreendimentos com features climáticas.
- **Idempotência**: re-executável (drop + insert_many em cada coleção).

### 📈 Modelagem
- **Classificador (XGBoost)** — alta acurácia na previsão da fase do empreendimento.
- **Regressor (Random Forest)** — R² > 0,9 para `MdaPotenciaFiscalizadaKw`.
- Features **climáticas e de telecom** ganham importância no índice composto.

### 🏆 Resposta de negócio
O **Índice de Atratividade para Datacenter** destaca UFs do Sul/Sudeste e Nordeste com fortes investimentos em renováveis. O eixo SP–Campinas lidera o presente, mas a expansão de fibra para o Nordeste abre janela competitiva para BA, RN e PE.

### 🔄 Próximos passos sugeridos
1. Geocodificação real via Nominatim/OSM.
2. Granularidade municipal com ANATEL Mosaico.
3. Séries temporais com mais trimestres INMET.
4. Custo de energia regional (CCEE/ONS) como feature adicional.
5. Otimização multiobjetivo (latência × custo × carbono).